# GeoTón Perú 2026 — Pipeline consolidado v2

**Categoría 3**: Desiertos de Servicios — accesibilidad a salud, educación y conectividad digital.

Este notebook orquesta la limpieza y merge de los **6 datasets** en una tabla maestra a nivel distrital (1,874 distritos).

## Módulos requeridos en `src/`

- `limpieza.py` — salud + distrital (pandas)
- `limpieza_escuelas.py` — escuelas MINEDU (Polars)
- `limpieza_osiptel.py` — cobertura móvil OSIPTEL (pandas)
- `limpieza_tambos.py` — Tambos PCM (pandas)
- `limpieza_ig.py` — Polígonos distritales INEI (Polars)

## Datos en `data/`

| Archivo | Contenido |
|---|---|
| `GeoPeru-peru_distritos(Perú Distritos).csv` | NBI + indicadores INEI 2017 |
| `GeoPeru-instituciones_salud_Peru Instituciones Salud.csv` | 9,254 EE.SS. |
| `GeoPeru-instituciones_educativas.xlsx` | Padrón MINEDU |
| `Cobertura móvil por empresa operadora.csv` | OSIPTEL 51k reportes |
| `Plataformas fijas TAMBOS prestando servicios I TRIMESTRE_2026.xlsx` | 521 Tambos |
| `ig_distrito.csv` | Polígonos distritales INEI |

## 0. Setup

In [9]:
from pathlib import Path
import sys

def find_project_root(start: Path):
    for path in [start, *start.parents]:
        if (path / "src").is_dir():
            return path
    raise FileNotFoundError("No se encontró la carpeta src")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\jrinc\Desktop\repositorio\GeoHaacaton


In [ ]:
# port sys
# from pathlib import Path
# import pandas as pd
# import polars as pl
# im
# sys.path.insert(0, str(Path('src').resolve()))

from utils.limpieza import (
    limpiar_salud, limpiar_distrital, features_salud_por_distrito
)
from utils.limpieza_escuelas import (
    limpiar_escuelas, features_escuelas_por_distrito, merge_escuelas_con_distrital
)
from utils.limpieza_osiptel import (
    limpiar_osiptel, features_osiptel_por_distrito, merge_osiptel_con_distrital
)
from utils.limpieza_tambos import (
    limpiar_tambos, features_tambos_por_distrito, merge_tambos_con_distrital
)
from utils.limpieza_ig import (
    limpiar_ig, calcular_centroides, merge_centroides_con_distrital
)

DATA = Path('../data')
OUT = Path('../output')
OUT.mkdir(exist_ok=True)
print('OK')

OK


## 1. Limpieza dataset por dataset

### 1.1 Distritos (INEI 2017) — TABLA BASE
Target: `pct_nbi`. Resultado: 1,874 × 42 cols.

In [15]:
df_dis = limpiar_distrital(DATA / 'GeoPeru-peru_distritos(Perú Distritos).csv')
df_dis.head()

[distrital] cargado:            1874 distritos × 109 cols
[distrital] tras corte:         1874 distritos × 42 cols
[distrital] cod_dist únicos:    1874
[distrital] nulos totales:         0
[distrital] media pct_nbi:      38.6%
[distrital] media pct_sin_int:  93.1%


,cod_dpto,nom_dpto,cod_prov,nom_prov,cod_dist,nom_dist,total_pers,num_hog,sup_tot,pct_nbi,...,pct_quintil3,pct_quintil4,pct_quintil5,pct_con_sis,pct_sin_seguro,pct_sin_documento,pct_no_sabe_leer,pct_pea_ocupada,pct_pea_desocupada,pct_pei
0,3,APURIMAC,302,ANDAHUAYLAS,30220,JOSE MARIA ARGUEDAS,4041.0,1.127,195.92,22.3,...,11.369762,9.115413,8.968390,89.659397,8.698848,0.098015,22.837540,34.991424,2.352365,30.090664
1,3,APURIMAC,304,AYMARAES,30415,TINTAY,2260.0,849.000,136.58,31.1,...,10.079225,7.086268,5.105634,85.871479,6.294014,0.132042,23.943662,56.954225,1.804577,17.297535
2,3,APURIMAC,302,ANDAHUAYLAS,30206,HUAYANA,708.0,262.000,96.87,20.2,...,10.933333,7.866667,4.933333,88.800000,4.533333,0.533333,18.266667,27.733333,2.400000,42.533333
3,3,APURIMAC,304,AYMARAES,30404,CHAPIMARCA,1803.0,727.000,213.09,60.7,...,9.297412,6.497623,5.124142,87.585843,6.656101,0.105652,24.088748,27.892235,3.433703,46.064448
4,3,APURIMAC,304,AYMARAES,30405,COLCABAMBA,683.0,268.000,95.75,9.4,...,9.956710,8.369408,6.637807,90.331890,4.329004,0.000000,20.634921,45.165945,5.916306,25.685426


### 1.2 Salud
8,967 EE.SS. activos, 192 hospitales (cat. II-1+).

In [17]:
df_sal = limpiar_salud(DATA / 'GeoPeru-instituciones_salud_Peru Instituciones Salud.csv')
df_sal.head()

[salud] inicial:                9254
[salud]   est_estado=ACTIVO:    9185
[salud]   est_condic=ACTIVO:    9188
[salud]   ctg_codigo != '0':    9023
[salud] tras filtros (AND):     8967  (96.9%)
[salud] hospitales (II-1+):      192
[salud] distritos con estab.:   1853 / 1,874


,cod_dist,nom_dist,cod_dpto,nom_dpto,cod_prov,nom_prov,est_nombre,clasificac,ctg_codigo,tipo_estab,est_estado,est_condic,internet,conect,lat,lon,es_hospital
0,250104,MASISEA,25,UCAYALI,2501,CORONEL PORTILLO,ISLA LIBERTAD,PUESTOS DE SALUD O POSTAS DE SALUD,I-1,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,ACTIVO,ACTIVO,NO,No tiene,-8.530365,-74.275027,False
1,250201,RAIMONDI,25,UCAYALI,2502,ATALAYA,CHICOSA,PUESTOS DE SALUD O POSTAS DE SALUD,I-1,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,ACTIVO,ACTIVO,NO,No tiene,-10.376356,-74.063880,False
2,250301,PADRE ABAD,25,UCAYALI,2503,PADRE ABAD,ALTO AGUAYTIA.,PUESTOS DE SALUD O POSTAS DE SALUD,I-1,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,ACTIVO,ACTIVO,NO,No tiene,-9.243097,-75.487659,False
3,250203,TAHUANIA,25,UCAYALI,2502,ATALAYA,BOLOGNESI,CENTROS DE SALUD O CENTROS MEDICOS,I-3,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,ACTIVO,ACTIVO,NO,No tiene,-10.026427,-73.951821,False
4,250201,RAIMONDI,25,UCAYALI,2502,ATALAYA,RIMA,PUESTOS DE SALUD O POSTAS DE SALUD,I-1,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,ACTIVO,ACTIVO,NO,No tiene,-10.730391,-73.595496,False


### 1.3 Escuelas (MINEDU)

In [18]:
df_esc = limpiar_escuelas(DATA / 'GeoPeru-instituciones_educativas.xlsx')
df_esc.head()

[escuelas] inicial:                  172750  (50 cols)
[escuelas] descartados por:
  sin cod_dist:                         176
  sin coordenadas:                        0
  sin nivel modular:                      0
  estado != Activo:                   57913
  formalidad != Escolarizado:         55723
  coordenadas fuera de Perú:           1366
[escuelas] tras filtros:              94049  (54.4%, 23 cols)
[escuelas] distritos cubiertos:        1874 / 1,874

[escuelas] por nivel:
  primaria:     39330
  secundaria:   15316
  inicial:      36380
  regulares:    88459
[escuelas] área rural:  50957 (54.2%)
[escuelas] públicas:    66339 (70.5%)


codmod,cod_local,cod_dist,nom_dist,cod_prov,nom_prov,cod_dpto,nom_dpto,lat,lon,cenedu,nivmod_val,esta_val,form_valor,gesdep_val,areaval,alt_cp,es_primaria,es_secundaria,es_inicial,es_regular,es_publico,area_rural
i64,i64,i64,str,i64,str,i64,str,f64,f64,str,str,str,str,str,str,i64,bool,bool,bool,bool,bool,bool
603654,469064,220101,"""MOYOBAMBA""",2201,"""MOYOBAMBA""",22,"""SAN MARTIN""",-6.03131,-76.96231,"""2""","""Básica Especial - Primaria""","""Activo""","""Escolarizado""","""Sector Educación""","""Urbana""",874,true,false,false,false,true,false
1738376,469064,220101,"""MOYOBAMBA""",2201,"""MOYOBAMBA""",22,"""SAN MARTIN""",-6.03131,-76.96231,"""2""","""Básica Especial - Inicial""","""Activo""","""Escolarizado""","""Sector Educación""","""Urbana""",874,false,false,true,false,true,false
761916,479789,220801,"""RIOJA""",2208,"""RIOJA""",22,"""SAN MARTIN""",-6.06211,-77.165,"""3""","""Básica Especial - Primaria""","""Activo""","""Escolarizado""","""Sector Educación""","""Urbana""",844,true,false,false,false,true,false
1738400,479789,220801,"""RIOJA""",2208,"""RIOJA""",22,"""SAN MARTIN""",-6.06211,-77.165,"""3""","""Básica Especial - Inicial""","""Activo""","""Escolarizado""","""Sector Educación""","""Urbana""",844,false,false,true,false,true,false
1379262,540249,220804,"""NUEVA CAJAMARCA""",2208,"""RIOJA""",22,"""SAN MARTIN""",-5.9302,-77.3102,"""00004 MARIA MONTESSORI""","""Básica Especial - Primaria""","""Activo""","""Escolarizado""","""Sector Educación""","""Urbana""",870,true,false,false,false,true,false


### 1.4 OSIPTEL
51,366 reportes CCPP×Empresa. 1,664 distritos con cobertura, 211 sin reporte.

In [19]:
df_ops = limpiar_osiptel(DATA / 'Cobertura móvil por empresa operadora.csv')
df_ops.head()

[osiptel] inicial:                     51366  (25 cols)
[osiptel] tras corte:                  51366  (15 cols)
[osiptel] período:                    202303
[osiptel] CCPP únicos:                 31439
[osiptel] distritos con reporte:        1664 / 1,874
[osiptel] operadoras:                 Viettel Perú S.A.C., Telefónica del Perú S.A.A., América Móvil Perú S.A.C., Entel Perú S.A.
[osiptel] cobertura por tecnología (reportes con valor=1):
   2G:    6341 (12.3%)
   3G:   35509 (69.1%)
   4G:   23048 (44.9%)
   5G:      30 (0.1%)


,cod_dist,cod_ccpp,departamento,provincia,distrito,centro_poblado,lat,lon,empresa,cob_2g,cob_3g,cob_4g,cob_5g,internet_rapido,n_estaciones_4g
0,10101,101010001,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,CHACHAPOYAS,-6.229414,-77.872497,Viettel Perú S.A.C.,0,1,1,0,1,5
1,10101,101010002,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,CACLIC,-6.202440,-77.901470,Viettel Perú S.A.C.,0,1,0,0,1,0
2,10101,101010004,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,EL TAPIAL,-6.208440,-77.867590,Viettel Perú S.A.C.,0,1,1,0,1,2
3,10101,101010006,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,POLLAPAMPA,-6.223390,-77.874460,Viettel Perú S.A.C.,0,1,1,0,1,2
4,10101,101010007,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,BOCANEGRA,-6.225290,-77.889250,Viettel Perú S.A.C.,0,1,0,0,1,0


### 1.5 Tambos (PCM)
521 plataformas en 407 distritos. **1,467 distritos sin Tambo** = oportunidad estatal.

In [20]:
df_tam = limpiar_tambos(DATA / 'Plataformas fijas TAMBOS prestando servicios I TRIMESTRE_2026.xlsx')
df_tam.head()

[tambos] inicial:                    521  (16 cols)
[tambos] tras corte:                 521  (14 cols)
[tambos] distritos con Tambo:        407 / 1,874
[tambos] departamentos:               22
[tambos] población total atendida:   1,021,364
[tambos] viviendas atendidas:          454,933


,cod_plataforma,cod_ccpp,cod_dist,nom_dpto,nom_prov,nom_dist,centro_poblado,nom_tambo,lat,lon,n_ccpp_influencia,pob_influencia,viv_influencia,fecha_inicio
0,202952,2106010090,210601,PUNO,HUANCANE,HUANCANE,HUARISANI,SAN PEDRO DE HUARISANI,-15.276335,-69.755085,27,2579,2308,20121219
1,207547,2101090066,210109,PUNO,PUNO,MAÑAZO,CHARAMAYA,CHARAMAYA,-15.984002,-70.502240,34,448,244,20121112
2,208293,807080008,80708,CUSCO,CHUMBIVILCAS,VELILLE,AYACCASI,AYACCASI,-14.417012,-71.886945,47,1050,392,20130929
3,208315,901050050,90105,HUANCAVELICA,HUANCAVELICA,CUENCA,LUQUIA,LUQUIA,-12.492532,-75.047197,49,780,642,20150907
4,208478,901010020,90101,HUANCAVELICA,HUANCAVELICA,HUANCAVELICA,SACSAMARCA,SACCSAMARCA,-12.799931,-74.992045,16,886,640,20121128


### 1.6 Polígonos distritales (ig_distrito)

Habilita:
- **Centroides** → KDTree → distancias `km_hospital`, `km_colegio`
- **Mapa coroplético** del Perú (lámina del jurado)

OJO: el WKT viene en orden **`lat lon`** (no estándar). El módulo lo maneja.

In [21]:
df_ig = limpiar_ig(DATA / 'ig_distrito.csv')
df_centroides = calcular_centroides(df_ig)
df_centroides.head()

[ig] inicial:            1874 × 14 cols
[ig] tras corte:         1874 × 6 cols
[ig] cod_dist únicos:    1874
[ig] departamentos:        25
[centroides] calculados:         1874
[centroides] rango lat:         [-18.23, -0.53]
[centroides] rango lon:         [-81.26, -68.89]


cod_dist,lat_centroide,lon_centroide
i64,f64,f64
130701,-7.456585,-79.375716
230302,-17.408933,-70.586201
130205,-7.722436,-79.304814
80806,-14.683377,-71.346751
101104,-9.717145,-76.568435


## 2. Agregaciones a nivel distrital

Cada dataset se enchufa a la tabla base.

In [22]:
df_full = df_dis.copy()
print(f'Base: {df_full.shape}')

Base: (1874, 42)


In [23]:
# Salud
df_full = features_salud_por_distrito(df_sal, df_full)
print(f'+ salud: {df_full.shape}')

[join] shape final:                  (1874, 46)
[join] distritos sin estab. salud:      21
[join] distritos sin hospital:        1716
[join] distritos con >= 1 hospital:    158
+ salud: (1874, 46)


In [24]:
# Escuelas
df_esc_agg = features_escuelas_por_distrito(df_esc)
df_full = merge_escuelas_con_distrital(df_full, df_esc_agg)
print(f'+ escuelas: {df_full.shape}')

[agg_escuelas] distritos con escuelas:  1874
[agg_escuelas] distritos sin secund.:     46
[agg_escuelas] distritos sin primaria:     0
+ escuelas: (1874, 56)


In [25]:
# OSIPTEL
df_ops_agg = features_osiptel_por_distrito(df_ops)
df_full = merge_osiptel_con_distrital(df_full, df_ops_agg)
print(f'+ osiptel: {df_full.shape}')

[osiptel_agg] distritos con reporte:     1664
[osiptel_agg] distritos sin 4G en ningún CCPP:   360
[osiptel_agg] media pct_ccpp_con_4g:     43.5%
[osiptel_agg] media operadores 4G/CCPP:  0.71
[merge_osiptel] shape final:                   (1874, 65)
[merge_osiptel] distritos sin reporte (= 0%):    211
[merge_osiptel] distritos con algo de 4G:       1304 (69.6%)
+ osiptel: (1874, 65)


C:\Users\jrinc\Desktop\repositorio\GeoHaacaton\src\limpieza_osiptel.py:256: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out['tiene_4g'] = out['tiene_4g'].fillna(False)


In [26]:
# Tambos
df_tam_agg = features_tambos_por_distrito(df_tam)
df_full = merge_tambos_con_distrital(df_full, df_tam_agg)
print(f'+ tambos: {df_full.shape}')

[tambos_agg] distritos con al menos 1 Tambo:  407
[tambos_agg] distritos SIN Tambo:           1467  ← oportunidad estatal
[tambos_agg] promedio Tambos por distrito:  1.28
[merge_tambos] shape final:                  (1874, 69)
[merge_tambos] distritos con Tambo:           407 (21.7%)
[merge_tambos] distritos SIN Tambo:          1467 (78.3%)
+ tambos: (1874, 69)


C:\Users\jrinc\Desktop\repositorio\GeoHaacaton\src\limpieza_tambos.py:143: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out['tiene_tambo'] = out['tiene_tambo'].fillna(False)


In [27]:
# Centroides distritales
df_full = merge_centroides_con_distrital(df_full, df_centroides)
print(f'+ centroides: {df_full.shape}')

[merge_ig] shape final:                  (1874, 71)
[merge_ig] distritos sin centroide:      0
+ centroides: (1874, 71)


## 3. Tabla maestra: validación y export

In [28]:
print(f'Shape final: {df_full.shape}')
print(f'Distritos: {df_full["cod_dist"].nunique()} (esperado 1,874)')
nulos = df_full.isna().sum()
if (nulos > 0).any():
    print(f'\nNulos:')
    print(nulos[nulos > 0].sort_values(ascending=False))
else:
    print('Sin nulos ✓')

Shape final: (1874, 71)
Distritos: 1874 (esperado 1,874)
Sin nulos ✓


In [29]:
# Top 10 candidatos a desierto múltiple: alta NBI + sin internet + sin Tambo
df_full.nlargest(10, 'pct_nbi')[[
    'nom_dpto', 'nom_dist', 'total_pers', 'pct_nbi', 'pct_sin_internet',
    'n_estab_salud', 'n_hospitales', 'pct_ccpp_con_4g', 'tiene_tambo'
]]

,nom_dpto,nom_dist,total_pers,pct_nbi,pct_sin_internet,n_estab_salud,n_hospitales,pct_ccpp_con_4g,tiene_tambo
1602,LORETO,CAHUAPANAS,6336.0,99.4,99.922601,7,0,0.0,True
1608,LORETO,ANDOAS,11569.0,96.4,99.787053,10,0,0.0,True
901,LIMA,TUPE,533.0,96.1,100.000000,1,0,0.0,False
557,CUSCO,KOSÑIPATA,4200.0,95.9,99.924471,2,0,0.0,False
1581,LORETO,BALSAPUERTO,13647.0,95.8,99.965963,20,0,0.0,True
1387,ANCASH,PARARIN,1517.0,95.6,99.803922,1,0,25.0,True
1566,AMAZONAS,RIO SANTIAGO,13778.0,94.2,99.934853,20,0,0.0,True
1246,UCAYALI,IPARIA,10227.0,92.7,99.871465,13,0,0.0,True
1611,LORETO,TORRES CAUSANA,4011.0,92.2,100.000000,5,0,0.0,True
1250,UCAYALI,MASISEA,11091.0,91.8,99.771603,25,0,37.5,True


In [31]:
df_full.to_parquet(OUT / 'results/features_distritales_v1.parquet', index=False)
df_full.to_csv(OUT / 'results/features_distritales_v1.csv', index=False)
print(f'Exportado a {OUT}/')

Exportado a ..\output/


## 4. Siguiente: distancias (KDTree)

Ya tenemos centroides + coordenadas de salud y escuelas. Construir KDTrees para:
- `km_hospital`: distancia al hospital más cercano (II-1+)
- `km_salud_cercano`: al EE.SS. más cercano
- `km_colegio` / `km_secundaria`: al colegio más cercano
- `n_estab_30km`: densidad dentro de 30 km

Eso cierra el **Bloque A**.

- **Bloque B**: LightGBM + SHAP para predecir `pct_nbi`
- **Bloque C**: Índice Alkire-Foster + lista priorizada